In [7]:
import json
import re
import random
from PIL import Image, ImageDraw, ImageFont
import textwrap
from tqdm import tqdm

In [6]:
from PIL import Image, ImageDraw, ImageFont

def wrap_text_by_pixel_width(text, font, max_width):
    """
    按像素宽度精确换行
    
    Args:
        text: 要换行的文本
        font: PIL字体对象
        max_width: 最大像素宽度
    
    Returns:
        lines: 换行后的文本列表
    """
    words = text.split()
    lines = []
    current_line = []
    current_width = 0
    
    for word in words:
        # 计算单词的实际像素宽度
        word_width = font.getbbox(word + " ")[2]
        
        # 如果加上这个单词会超出宽度,换行
        if current_width + word_width > max_width and current_line:
            lines.append(" ".join(current_line))
            current_line = [word]
            current_width = word_width
        else:
            current_line.append(word)
            current_width += word_width
    
    # 添加最后一行
    if current_line:
        lines.append(" ".join(current_line))
    
    return lines

def render_text_fixed_width(text, font_path, font_size=16, 
                           width=900, padding=20, line_spacing=4):
    """
    固定宽度,高度自适应,精确像素换行
    """
    # 加载字体
    font = ImageFont.truetype(font_path, font_size)
    
    # 计算可用文本宽度
    text_width = width - 2 * padding
    
    # ✅ 使用像素宽度精确换行
    lines = wrap_text_by_pixel_width(text, font, text_width)
    
    # 计算总高度
    total_height = padding
    for line in lines:
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        total_height += line_height + line_spacing
    total_height = total_height - line_spacing + padding
    
    # 创建图片
    img = Image.new("RGB", (width, total_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # 绘制文本
    y_offset = padding
    for line in lines:
        draw.text((padding, y_offset), line, font=font, fill=(0, 0, 0))
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        y_offset += line_height + line_spacing
    
    return img

In [2]:
def replace_random_words_with_stars(text, replace_percent=5):
    # 正则匹配单词：字母和可能的撇号组合（如 don't, FREEDOM, a, it's）
    word_pattern = re.compile(r"\b[\w']+\b")
    
    # 找出所有单词及其位置信息
    words = list(word_pattern.finditer(text))
    total_words = len(words)
    
    if total_words == 0:
        return text  # 没有单词，直接返回原文

    # 计算需要替换的单词数量（约 5%）
    num_to_replace = max(1, round(total_words * replace_percent / 100))  # 至少替换1个，避免0
    
    # 随机选择要替换的单词的索引
    replace_indices = random.sample(range(total_words), num_to_replace)
    
    # 构建新的文本
    text_list = list(text)
    for idx in sorted(replace_indices, reverse=True):  # 逆序处理，避免影响后续索引
        match = words[idx]
        start, end = match.span()
        # 替换单词为 **
        text_list[start:end] = '**'
    
    return ''.join(text_list)

In [3]:
with open("../fox_data/data.json","r") as f:
    data = json.load(f)

In [4]:
for item in data:
    text = item["gt_text"]
    replace_text = replace_random_words_with_stars(text)
    item["replace"] = replace_text

In [5]:
with open("../fox_data/replace.json","w") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [8]:
# 使用
for item in tqdm(data):
    img = render_text_fixed_width(
        text=item["replace"],
        font_path="../fonts/NotoSans-Regular.ttf",
        font_size=16,
        width=900,
        padding=20,
        line_spacing=4
    )
    img.save(f"../fox_data/replace/{item['image']}")

100%|██████████| 112/112 [06:36<00:00,  3.54s/it]
